<h1 style=\"text-align: center; font-size: 50px;\"> <h1 style=\"text-align: center; font-size: 50px;\"> 📦 Register Model </h1> </h1>

This notebook packages the **audio-native agentic workflow** as an **MLflow pyfunc model**, logs it with artifacts
(index, config), and registers it to the MLflow Model Registry for serving.

- Retrieval: **CLAP** audio↔text embeddings over timestamped audio windows (+ **MMR** reranker)
- Generation: **Qwen Omni** listens to the selected audio windows and answers (no transcripts required)
- Orchestration: **LangGraph** (relevance → memory → retrieve → rerank → answer → memoize)
- Vector store: **FAISS** (in-model artifact or built on first run)
- Memory: disk-backed key-value cache (per-corpus+question)


# Notebook Overview

- Start Execution
- Define User Constants
- Install and Import Libraries
- Configure Settings
- Verify Assets
- KV Memory
- LLM Setup
- State Model
- Node Functions
- Graph Definition
- Graph Visualization
- Generated Answer
- Message History

# Start Execution

In [1]:
# Standard library imports
import os  # Provides OS-related utilities
import sys  # Allows manipulation of Python runtime environment
import time  # Enables time-based operations
from pathlib import Path  # Object-oriented file system paths

# Extend sys.path to allow importing from parent directory
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from src.model_selection import ModelSelector
from src.simple_kv_memory import SimpleKVMemory  # In-memory key-value store for agent state
from src.utils import (  # Utility functions for logging, LLM I/O, and schema generation
    load_config,
    load_secrets,
    load_secrets_to_env,
    configure_proxy,
    display_image,
    get_project_root,
    get_response_from_llm,
    json_schema_from_type,
    log_timing,
    sec_to_timestamp,
    logger,
    login_huggingface,
    setup_model_environment,
    ensure_wav,
)

In [2]:
start_time = time.time()  
logger.info("Notebook execution started.")

# Define User Constants

In [3]:
QUESTION: str = "What is the main idea of the content?"
DOCS: list
FILE_ID: str
MEMORY: SimpleKVMemory
INPUT_PATH: Path = Path("../data/input")

# Install and Import Libraries

In [4]:
%%time

%pip install -r ../requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.
CPU times: user 29.1 ms, sys: 20.6 ms, total: 49.7 ms
Wall time: 1.84 s


In [5]:
from __future__ import annotations  # Enables postponed evaluation of annotations (PEP 563)

# ─────── Standard Library ───────
import base64  # Provides encoding and decoding of binary data
import functools  # Higher-order functions for functional programming 
import json  # JSON serialization and deserialization
import logging  # Flexible logging system
import multiprocessing  # Support for spawning processes
import shutil  # High-level file operations
import warnings  # Issue warning messages
from collections import namedtuple  # Factory for creating tuple subclasses with named fields
from datetime import datetime  # Date and time utilities
from pathlib import Path  # Object-oriented filesystem paths
from typing import Any, Dict, List, Literal, Optional, TypedDict  # Type hinting support
import numpy as np  # Numerical operations and array handling
import soundfile as sf

from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq


# ─────── Third-Party Packages ───────
import mlflow  # Model tracking and serving framework
from mlflow.tracking import MlflowClient  # Interface to interact with MLflow tracking server for experiments, runs, and artifacts
import yaml  # YAML parsing and serialization
from IPython.display import Markdown, display  # IPython utilities for notebook output formatting
from tqdm import tqdm  # Visual progress bar for loops
import torch # PyTorch for tensor computations and deep learning
import torchaudio  # PyTorch audio processing library
from huggingface_hub import snapshot_download, hf_hub_download # Hugging Face Hub utilities for model and dataset management
import soundfile as sf  # Library for reading and writing sound files
import faiss # Library for efficient similarity search and clustering of dense vectors

# Qwen Omni (audio+video+text) – both full & Thinker-only variants
from transformers import Qwen2_5OmniProcessor, Qwen2_5OmniThinkerForConditionalGeneration # Processor and model for Qwen Omni
from transformers import AutoProcessor as ClapProcessor, ClapModel # CLAP processor and model for audio embeddings
from qwen_omni_utils import process_mm_info     # official utils to prep audio/video inputs

# ─────── LangChain Core & Community ───────
from langchain.docstore.document import Document  # Core document abstraction
from langchain_community.llms import LlamaCpp  # Integration for local LlamaCpp models
from langgraph.graph import StateGraph, END

from src.agentic_workflow import build_agentic_graph
from src.agentic_audio_rag_model import AgenticAudioRAGModel  # Core agent logic for audio RAG tasks
from src.simple_kv_memory import SimpleKVMemory  # In-memory key-value store for agent state

# Qwen Omni adapter you already defined in run-workflow (repeat here if not in src)
# If your Qwen adapter class lives in src, import it; else we redefine below.
try:
    from agentic_audio_rag_model import QwenOmniAgent  # example path if you saved it
except Exception:
    QwenOmniAgent = None  # we'll define a minimal version inside the pyfunc model if missing


# Configure Settings

In [6]:
# Suppress Python warnings
warnings.filterwarnings("ignore")

In [7]:
project_root = get_project_root()
INPUT_PATH: Path = Path("../data/input")  

MEMORY_PATH: Path = Path("../data/memory")
CONFIG_PATH = "../configs/config.yaml"
SECRETS_PATH = "../configs/secrets.yaml"

# LLAMA_MODEL_PATH = "/home/jovyan/datafabric/meta-llama3.1-8b-Q8/Meta-Llama-3.1-8B-Instruct-Q8_0.gguf"
SAMPLE_MEDIA_PATH = INPUT_PATH / "sample_tts.mp3"
CONTEXT_WINDOW = 8192
MAX_TOKENS = CONTEXT_WINDOW // 8
CHUNK_SIZE = CONTEXT_WINDOW // 2
CHUNK_OVERLAP = CHUNK_SIZE // 8  

EXPERIMENT_NAME = "AIStudio-Agentic-Audio-RAG-with-LangGraph-Experiment"
RUN_NAME = "AIStudio-Agentic-Audio-RAG-with-LangGraph-Run"
MODEL_NAME = "AIStudio-Agentic-Audio-RAG-with-LangGraph-Model"

# --- Retrieval / Rerank params (must match run-workflow) ---
MEMORY_FILENAME = "kv_memory.jsonl"
INDEX_VECS_NPY = "audio_vecs.npy"
INDEX_META_JSON = "audio_meta.json"
RELEVANCE_THRESHOLD = 0.18
FETCH_K = 24     # breadth for stage-1
TOP_K   = 6      # final segments

In [8]:
# Load secrets from secrets.yaml file (if it exists) into environment
if Path(SECRETS_PATH).exists():
    load_secrets_to_env(SECRETS_PATH)
else:
    print(f"No secrets file found at {SECRETS_PATH}; relying on preexisting environment")

# Retrieve secrets from environment
try:
    secrets = load_secrets()
except ValueError:
    secrets = {}

# Load configuration and secrets
config = load_config(CONFIG_PATH)

print("✅ Configuration loaded successfully")
print("✅ Secrets loaded successfully")

✅ Loaded 1 secrets into environment variables.
✅ Configuration loaded successfully
✅ Secrets loaded successfully


In [9]:
logger.info('Notebook execution started.')

## Verify Assets

In [10]:
def log_asset_status(asset_path: str, asset_name: str) -> None:
    """
    Logs the status of a given asset based on its existence.

    Parameters:
        asset_path (str): File or directory path to check.
        asset_name (str): Name of the asset for logging context.
    """
    if Path(asset_path).exists():
        logger.info(f"{asset_name} is properly configured.")
    else:
        logger.info(f"{asset_name} is not properly configured. Please ensure the required asset is correctly configured in your AI Studio project according to the README file.")

def log_secrets_status(secrets: Dict[str, Any], success_message: str, failure_message: str) -> None:
    """
    Logs the status of secrets based on their existence.

    Parameters:
        secrets (Dict[str, Any]): Secrets retrieved to check if they exist.
        success_message (str): Message to log if secrets exists.
        failure_message (str): Message to log if secrets do not exist.
    """
    if secrets:
        logger.info(f"Project secrets are available. {success_message}")
    else:
        logger.info(f"There are no project secrets found. {failure_message}")

In [11]:
log_asset_status(
    asset_path=INPUT_PATH,
    asset_name="Input Data",
)

# log_asset_status(
#     asset_path=LLAMA_MODEL_PATH,
#     asset_name="LLM",
# )

log_asset_status(
    asset_path=CONFIG_PATH,
    asset_name="Config",
)

log_secrets_status(
    secrets=secrets,
    success_message="",
    failure_message="Please check if the secrets were propely connfigured in your secrets yaml file or in Secrets Manager."
)

# KV Memory

In [12]:
memory: SimpleKVMemory = SimpleKVMemory(MEMORY_PATH)
memory.set('dummy key', 'dummy value')

# Prepare Audio Files for Inference

In [14]:
logger.info("🎧 Scanning directory for media files: %s", INPUT_PATH)

# Prefer an audio-native LLM
AUDIO_LLM = {
 #   "MiDaSheng": ("MiSpeech/MiDaShengLM-7B-GGUF"),
 #   "Kimi": ("Moonshot-AI/Kimi-Audio-7B-Instruct"),
    "Qwen": ("Qwen/Qwen2.5-Omni-7B"), 
}["Qwen"]

# Supported media types
AUDIO_EXTS = {".mp3", ".wav", ".ogg", ".flac", ".m4a"}
VIDEO_EXTS = {".mp4", ".mov", ".avi", ".mkv"}
MEDIA_EXTS = AUDIO_EXTS | VIDEO_EXTS

SRC_DIR   = project_root / "src"
DATA_DIR  = project_root / "data"
INPUT_DIR = DATA_DIR / "input"     # media to index (same as run-workflow)
ARTIF_DIR = project_root / "artifacts" # temp artifacts for logging

# Make src importable
sys.path.insert(0, str(SRC_DIR))

print("Project root:", project_root)
print("Src dir    :", SRC_DIR)
print("Data dir   :", DATA_DIR)
print("Inputs     :", INPUT_DIR)
print("Artifacts  :", ARTIF_DIR)

# Ensure HF cache paths live in the project area (matches README/setup)
setup_model_environment()

Project root: /home/jovyan/AI-Blueprints/generative-ai/agentic-audio-rag-with-langgraph
Src dir    : /home/jovyan/AI-Blueprints/generative-ai/agentic-audio-rag-with-langgraph/src
Data dir   : /home/jovyan/AI-Blueprints/generative-ai/agentic-audio-rag-with-langgraph/data
Inputs     : /home/jovyan/AI-Blueprints/generative-ai/agentic-audio-rag-with-langgraph/data/input
Artifacts  : /home/jovyan/AI-Blueprints/generative-ai/agentic-audio-rag-with-langgraph/artifacts


In [15]:

# def _ensure_hf_local(model_id: str) -> str:
#     """
#     Ensure the model exists locally; download snapshot if needed.
#     Uses project-local cache path derived by utils.format_model_path().
#     """
#     selector = ModelSelector()
#     local_dir = selector.format_model_path(model_id)
#     local_dir.mkdir(parents=True, exist_ok=True)
#     # Download only if the directory is empty
#     if not any(local_dir.iterdir()):
#         logger.info("⬇️ Downloading audio LLM '%s' to %s", model_id, str(local_dir))
#         snapshot_download(
#             repo_id=model_id,
#             local_dir=str(local_dir),
#             local_dir_use_symlinks=False,
#             resume_download=True,
#         )
#     return str(local_dir)

# def transcribe_with_audio_llm(media_path: str, model_id: str) -> Dict[str, Any]:
#     """
#     Transcribe an audio/video file using an audio-native LLM (no Whisper).
#     Returns {"text": <full transcript>, "segments": [{"start": float, "end": float, "text": str}, ...]}
#     """
#     device = "cuda" if torch.cuda.is_available() else "cpu"
#     local_dir = _ensure_hf_local(model_id)

#     # Load processor + model
#     processor = AutoProcessor.from_pretrained(local_dir, trust_remote_code=True)
#     model = AutoModelForSpeechSeq2Seq.from_pretrained(
#         local_dir,
#         torch_dtype=torch.float16 if device == "cuda" else torch.float32,
#         trust_remote_code=True,
#     ).to(device)

#     # Convert to wav (mono, 16 kHz) when needed
#     wav_path = ensure_wav(media_path)

#     # Try processor-native loader first, then soundfile
#     try:
#         audio, sr = processor.audio_load(wav_path)  # some processors expose this
#     except Exception:
#         audio, sr = soundfile.read(wav_path)

#     inputs = processor(audio=audio, sampling_rate=sr, return_tensors="pt").to(device)

#     with torch.no_grad():
#         generated = model.generate(**inputs, max_new_tokens=8192)

#     text = processor.batch_decode(generated, skip_special_tokens=True)[0].strip()

#     # If the model doesn’t return diarized/segmented timestamps,
#     # provide a single coarse segment as a placeholder.
#     # Downstream chunking will enrich this into overlapping windows.
#     est_seconds = max(10.0, len(text.split()) / 2.5)
#     segments = [{"start": 0.0, "end": float(est_seconds), "text": text}]

#     return {"text": text, "segments": segments}

# # -------- Scan & ingest media --------
# all_docs: List[Document] = []

# media_files = []
# for file_path in Path(INPUT_PATH).rglob("*"):
#     # Skip hidden/system folders
#     if any(part.startswith(".") and part not in {".", ".."} for part in file_path.parts):
#         continue
#     if file_path.suffix.lower() in MEDIA_EXTS:
#         media_files.append(file_path)

# if not media_files:
#     logger.warning("📭 No audio/video files found in %s", INPUT_PATH)

# for media_path in media_files:
#     try:
#         logger.info("🔊 Transcribing: %s  (model=%s)", media_path.name, AUDIO_LLM)
#         result = transcribe_with_audio_llm(str(media_path), AUDIO_LLM)

#         transcript_text = result["text"]
#         segments = result.get("segments", [])

#         # Wrap as a single LangChain Document (full transcript).
#         # Timestamps are preserved in metadata for downstream UI/reranker.
#         doc = Document(
#             page_content=transcript_text,
#             metadata={
#                 "file_path": str(media_path),
#                 "file_name": media_path.name,
#                 "media_type": "audio" if media_path.suffix.lower() in AUDIO_EXTS else "video",
#                 "segments": segments,  # [{"start": float, "end": float, "text": str}, ...]
#                 "source": "audio_llm_transcription",
#                 "audio_llm": AUDIO_LLM,
#             },
#         )
#         all_docs.append(doc)
#         logger.info("✅ Loaded transcript as Document: %s (chars=%d)", media_path.name, len(transcript_text))
#     except Exception as e:
#         logger.warning("❌ Failed to transcribe %s: %s", media_path.name, e)

# logger.info("📦 Total media files processed: %d — Documents created: %d", len(media_files), len(all_docs))


# INPUT_TEXT = '\n\n'.join([doc.page_content for doc in all_docs])

In [38]:
# Reuse the CLAP init + embedding utilities from your run-workflow notebook
# If you already defined them earlier in this kernel, skip redefining.
from transformers import ClapProcessor, ClapModel
import torch

# CLAP init
CLAP_REPO = "laion/clap-htsat-unfused"
clap_device = "cuda" if torch.cuda.is_available() else "cpu"
clap_processor = ClapProcessor.from_pretrained(CLAP_REPO)
clap_model = ClapModel.from_pretrained(CLAP_REPO).to(clap_device).eval()

try:
    clap_model.to("cpu")
    clap_device = "cpu"
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    print("CLAP moved to CPU; GPU cache cleared")
except Exception as e:
    print("Skipping CLAP offload:", e)

def _resample_to_48k(wav: np.ndarray, sr: int, target_sr: int = 48000) -> np.ndarray:
    if sr == target_sr:
        return wav.astype(np.float32, copy=False)
    try:
        import torchaudio
        t = torch.as_tensor(wav, dtype=torch.float32).unsqueeze(0)
        t48 = torchaudio.functional.resample(t, sr, target_sr)
        return t48.squeeze(0).cpu().numpy().astype(np.float32)
    except Exception:
        x = np.linspace(0, 1, num=wav.shape[0], dtype=np.float64, endpoint=False)
        y = np.interp(np.linspace(0, 1, num=int(round(wav.shape[0] * target_sr / sr)), endpoint=False),
                      x, wav.astype(np.float64, copy=False))
        return y.astype(np.float32)

@torch.no_grad()
def clap_embed_audio(wav: np.ndarray, sr: int) -> np.ndarray:
    wav48 = _resample_to_48k(wav, sr, 48000)
    inp = clap_processor(audios=[wav48], sampling_rate=48000, return_tensors="pt").to(clap_device)
    out = clap_model.get_audio_features(**inp)
    vec = out.cpu().numpy()[0]
    vec = vec / (np.linalg.norm(vec) + 1e-12)
    return vec.astype(np.float32)

@torch.no_grad()
def clap_embed_text(query: str) -> np.ndarray:
    inp = clap_processor(text=[query], return_tensors="pt").to(clap_device)
    out = clap_model.get_text_features(**inp)
    vec = out.cpu().numpy()[0]
    vec = vec / (np.linalg.norm(vec) + 1e-12)
    return vec.astype(np.float32)

# Segmentation (same as run-workflow)
from typing import List, Tuple, Dict, Any
def segment_audio(wav_path: str, window_s: float = 30.0, hop_s: float = 15.0) -> List[Tuple[int, int, np.ndarray, int]]:
    audio, sr = sf.read(wav_path)
    if audio.ndim == 2:
        audio = audio.mean(axis=1)
    if audio.dtype != np.float32:
        audio = audio.astype(np.float32)
    n = len(audio); win = int(window_s * sr); hop = int(hop_s * sr)
    if n == 0: return []
    segs, i = [], 0
    while i < n:
        j = min(i + win, n)
        segs.append((i, j, audio[i:j], sr))
        if j == n: break
        i += hop
    return segs

# In-memory FAISS index shell
class AudioIndex:
    def __init__(self, dim: int = 512):
        self.index = faiss.IndexFlatIP(dim)
        self.meta: List[Dict[str, Any]] = []
    def add(self, vecs: np.ndarray, metas: List[Dict[str, Any]]):
        # cosine via normalized IP
        vecs = vecs / (np.linalg.norm(vecs, axis=1, keepdims=True) + 1e-12)
        self.index.add(vecs.astype(np.float32))
        self.meta.extend(metas)
    def search(self, qvec: np.ndarray, k: int = 6) -> List[Dict[str, Any]]:
        qvec = qvec.astype(np.float32)
        qvec = qvec / (np.linalg.norm(qvec) + 1e-12)
        D, I = self.index.search(qvec[np.newaxis, :], k)
        out = []
        for idx, score in zip(I[0], D[0]):
            if 0 <= idx < len(self.meta):
                m = dict(self.meta[idx]); m["score"] = float(score)
                out.append(m)
        return out

# Build index from INPUT_DIR and snapshot to artifacts/
AUDIO_EXTS = {".mp3", ".wav", ".ogg", ".flac", ".m4a"}
VIDEO_EXTS = {".mp4", ".mov", ".avi", ".mkv", ".m4v", ".webm"}
MEDIA_EXTS = AUDIO_EXTS | VIDEO_EXTS

ARTIF_DIR.mkdir(parents=True, exist_ok=True)
(ARTIF_DIR / "index").mkdir(parents=True, exist_ok=True)
(ARTIF_DIR / "config").mkdir(parents=True, exist_ok=True)
(ARTIF_DIR / "memory").mkdir(parents=True, exist_ok=True)  # empty initial memory

# Collect media
media_paths = []
for p in sorted(Path(INPUT_DIR).rglob("*")):
    if any(part.startswith(".") and part not in {".", ".."} for part in p.parts):
        continue
    if p.is_file() and p.suffix.lower() in MEDIA_EXTS:
        media_paths.append(p)

# Embed segments
audio_index = AudioIndex(dim=512)
for media_path in media_paths:
    wav_path = ensure_wav(str(media_path))
    segs = segment_audio(wav_path, window_s=30.0, hop_s=15.0)
    if not segs: continue
    vecs, metas = [], []
    for (s0, s1, wav_seg, sr) in segs:
        v = clap_embed_audio(wav_seg, sr); vecs.append(v)
        metas.append({
            "file_path": str(media_path),
            "file_name": media_path.name,
            "wav_path": wav_path,
            "start_s": float(s0 / sr),
            "end_s": float(s1 / sr),
        })
    audio_index.add(np.stack(vecs, axis=0), metas)

# Persist index vectors + metadata as model artifacts
# We need the raw (already normalized) vectors; FAISS can't be pickled easily across runtimes.
# Re-run a pass to collect vectors in the same order FAISS used:
# (For simplicity, we re-embed here; for large corpora, persist as you add)
vecs = []
for m in audio_index.meta:
    audio, sr = sf.read(m["wav_path"])
    if audio.ndim == 2:
        audio = audio.mean(axis=1)
    i0 = int(m["start_s"] * sr); i1 = int(m["end_s"] * sr)
    wav_seg = audio[i0:i1].astype(np.float32, copy=False)
    vecs.append(clap_embed_audio(wav_seg, sr))
vecs = np.stack(vecs, axis=0).astype(np.float32)
np.save(ARTIF_DIR / "index" / INDEX_VECS_NPY, vecs)

with open(ARTIF_DIR / "index" / INDEX_META_JSON, "w") as f:
    json.dump(audio_index.meta, f, ensure_ascii=False, indent=2)

# Write a simple runtime config
config = {
    "relevance_threshold": RELEVANCE_THRESHOLD,
    "fetch_k": FETCH_K,
    "top_k": TOP_K,
    "clap_repo": CLAP_REPO,
    "media_root": str(INPUT_DIR),
}
with open(ARTIF_DIR / "config" / "config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Indexed segments:", len(audio_index.meta))


CLAP moved to CPU; GPU cache cleared
Indexed segments: 22


In [39]:
def _extract_window(wav_path: str, start_s: float, end_s: float) -> tuple[np.ndarray, int]:
    audio, sr = sf.read(wav_path)
    if audio.ndim == 2:
        audio = audio.mean(axis=1)
    i0 = max(0, int(start_s * sr)); i1 = max(i0, int(end_s * sr))
    return audio[i0:i1].astype(np.float32, copy=False), sr

def rerank_hits_mmr(query: str, hits: list[dict], top_k: int = 6, fetch_k: int = 24, lam: float = 0.6) -> list[dict]:
    if not hits: return []
    cands = hits[:max(fetch_k, top_k)]
    qvec = clap_embed_text(query); qvec = qvec / (np.linalg.norm(qvec) + 1e-12)
    cand_vecs = []
    for h in cands:
        wav_seg, sr = _extract_window(h["wav_path"], h["start_s"], h["end_s"])
        if wav_seg.size == 0:
            cand_vecs.append(None); continue
        v = clap_embed_audio(wav_seg, sr)
        cand_vecs.append(v / (np.linalg.norm(v) + 1e-12))
    kept = [(i, h, v) for i, (h, v) in enumerate(zip(cands, cand_vecs)) if v is not None]
    if not kept: return hits[:top_k]
    idxs, cands, cand_vecs = zip(*kept)
    cand_vecs = np.stack(cand_vecs, axis=0)

    chosen, chosen_idx, avail = [], [], set(range(len(cands)))
    while avail and len(chosen) < min(top_k, len(cands)):
        best_i, best_score = None, -1e9
        for i in avail:
            rel = float(np.dot(qvec, cand_vecs[i]))
            div = 0.0 if not chosen_idx else max(float(np.dot(cand_vecs[i], cand_vecs[j])) for j in chosen_idx)
            score = lam * rel - (1.0 - lam) * div
            if score > best_score:
                best_i, best_score = i, score
        item = dict(cands[best_i]); item["score_mmr"] = float(best_score)
        chosen.append(item); chosen_idx.append(best_i); avail.remove(best_i)
    return chosen

def retrieve_audio_segments_from_artifact(query: str, vecs: np.ndarray, metas: list[dict], k: int = 24) -> list[dict]:
    # fast search via FAISS flat IP (rebuild small index on the fly)
    idx = faiss.IndexFlatIP(vecs.shape[1])
    normed = vecs / (np.linalg.norm(vecs, axis=1, keepdims=True) + 1e-12)
    idx.add(normed.astype(np.float32))
    qvec = clap_embed_text(query); qvec = qvec / (np.linalg.norm(qvec) + 1e-12)
    D, I = idx.search(qvec[np.newaxis, :].astype(np.float32), k)
    out = []
    for i, score in zip(I[0], D[0]):
        if 0 <= i < len(metas):
            m = dict(metas[i]); m["score"] = float(score); out.append(m)
    return out


In [40]:
from typing import TypedDict, Optional, List, Dict, Any
from typing import Annotated
import operator

Messages = Annotated[List[Dict[str, Any]], operator.add]

class AudioState(TypedDict, total=False):
    question: str
    file_id: str
    memory: Any
    audio_llm: Any

    is_relevant: bool
    from_memory: bool
    hits_raw: List[Dict[str, Any]]
    hits: List[Dict[str, Any]]
    evidence: List[Dict[str, Any]]
    answer: str

    messages: Messages

def build_audio_agentic_graph_no_textllm(relevance_threshold: float, fetch_k: int, top_k: int,
                                         vecs: np.ndarray, metas: list[dict]):
    def _mem_get(mem, key):
        if isinstance(mem, dict): return mem.get(key)
        return mem.get(key) if hasattr(mem, "get") else None
    def _mem_put(mem, key, value):
        if isinstance(mem, dict):
            mem[key] = value; return
        if hasattr(mem, "set"): mem.set(key, value); return
        if hasattr(mem, "put"): mem.put(key, value); return
        raise RuntimeError("Unsupported memory object (no set/put)")

    def node_ingest_question(state: AudioState) -> AudioState:
        q = (state.get("question") or "").strip()
        if not q: raise ValueError("Empty question")
        return {"messages": [{"role":"developer","content":"Ingested question"},
                             {"role":"user","content": q}]}

    def node_check_relevance_audio(state: AudioState) -> AudioState:
        q = state["question"]
        probe = retrieve_audio_segments_from_artifact(q, vecs, metas, k=8)
        max_score = max([h.get("score", 0.0) for h in probe], default=0.0)
        is_rel = bool(max_score >= relevance_threshold)
        updates: AudioState = {"is_relevant": is_rel,
                               "messages": [{"role":"developer","content": f"Relevance: max_score={max_score:.3f}->{'relevant' if is_rel else 'irrelevant'}"}]}
        if not is_rel:
            updates["answer"] = "🚫 Sorry, I can’t find anything relevant to that question in this media."
        return updates

    def node_check_memory(state: AudioState) -> AudioState:
        q = (state.get("question") or "").strip().lower()
        fid = state.get("file_id", "global")
        key = f"{fid} :: {q}"
        cached = _mem_get(state.get("memory"), key)
        if cached:
            return {"from_memory": True,
                    "answer": cached.get("answer",""),
                    "evidence": cached.get("evidence", []),
                    "messages": [{"role":"developer","content": f"Cache hit for {key}"}]}
        else:
            return {"from_memory": False,
                    "messages": [{"role":"developer","content": f"Cache miss for {key}"}]}

    def node_retrieve(state: AudioState) -> AudioState:
        hits_raw = retrieve_audio_segments_from_artifact(state["question"], vecs, metas, k=fetch_k)
        return {"hits_raw": hits_raw}

    def node_rerank(state: AudioState) -> AudioState:
        hits = rerank_hits_mmr(state["question"], state.get("hits_raw", []), top_k=top_k, fetch_k=fetch_k, lam=0.6)
        return {"hits": hits}

    def node_generate_audio_only(state: AudioState) -> AudioState:
        hits = state.get("hits", [])
        llm = state.get("audio_llm")
        if llm is None:
            raise RuntimeError("Missing `audio_llm` (QwenOmniAgent). Pass it in the graph state.")
        out = llm.answer(state["question"], hits, return_audio=False)
        ev = out.get("evidence", [])
        for e in ev:
            if "score_mmr" in e:
                e["score"] = e["score_mmr"]
        return {"answer": out.get("answer",""), "evidence": ev}

    def node_update_memory(state: AudioState) -> AudioState:
        q = state["question"].strip().lower()
        fid = state.get("file_id", "global")
        key = f"{fid} :: {q}"
        # write
        mem = state.get("memory")
        if mem is not None:
            val = {"answer": state.get("answer",""), "evidence": state.get("evidence", [])}
            _mem_put(mem, key, val)
        return {}

    def node_output(state: AudioState) -> AudioState:
        return {}

    g = StateGraph(AudioState)
    g.add_node("ingest_question", node_ingest_question)
    g.add_node("check_relevance_audio", node_check_relevance_audio)
    g.add_node("check_memory", node_check_memory)
    g.add_node("retrieve", node_retrieve)
    g.add_node("rerank", node_rerank)
    g.add_node("generate_audio", node_generate_audio_only)
    g.add_node("update_memory", node_update_memory)
    g.add_node("output_answer", node_output)

    g.set_entry_point("ingest_question")
    g.add_edge("ingest_question", "check_relevance_audio")

    def after_relevance(state: AudioState):
        return "check_memory" if state.get("is_relevant") else "output_answer"
    g.add_conditional_edges("check_relevance_audio", after_relevance,
                            {"check_memory":"check_memory", "output_answer":"output_answer"})

    def after_memory(state: AudioState):
        return "output_answer" if state.get("from_memory") else "retrieve"
    g.add_conditional_edges("check_memory", after_memory,
                            {"output_answer":"output_answer", "retrieve":"retrieve"})

    g.add_edge("retrieve", "rerank")
    g.add_edge("rerank", "generate_audio")
    g.add_edge("generate_audio", "update_memory")
    g.add_edge("update_memory", "output_answer")
    g.add_edge("output_answer", END)

    return g.compile()


# MLflow Registration

In [ ]:
import mlflow
import mlflow.pyfunc
import pandas as pd

# SimpleKVMemory shim (uses .get/.set under the hood)
try:
    from simple_kv_memory import SimpleKVMemory as _MemClass
except Exception:
    from simple_kv_memory import SimpleKVMem as _MemClass

# Global memory-friendly knobs
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_grad_enabled(False)


class AudioAgenticPyFunc(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        setup_model_environment()

        # --- Artifacts
        index_dir   = Path(context.artifacts["index_dir"])
        config_path = Path(context.artifacts["config_path"])
        memory_dir  = Path(context.artifacts["memory_dir"])
        memory_dir.mkdir(parents=True, exist_ok=True)

        vecs_name = globals().get("INDEX_VECS_NPY", "audio_vecs.npy")
        meta_name = globals().get("INDEX_META_JSON", "audio_meta.json")

        self.vecs  = np.load(index_dir / vecs_name).astype(np.float32)
        with open(index_dir / meta_name, "r") as f:
            self.metas = json.load(f)
        with open(config_path, "r") as f:
            cfg = json.load(f)

        self.relevance_threshold = float(cfg.get("relevance_threshold", 0.18))
        self.fetch_k = int(cfg.get("fetch_k", 24))
        self.top_k   = int(cfg.get("top_k", 6))

        # --- CLAP on CPU (save VRAM for Qwen)
        global clap_processor, clap_model
        if "clap_processor" not in globals() or "clap_model" not in globals():
            clap_processor = ClapProcessor.from_pretrained(cfg["clap_repo"])
            clap_model = ClapModel.from_pretrained(cfg["clap_repo"]).eval()
            try:
                clap_model.to("cpu")
            except Exception:
                pass

        # --- Memory (Path, not str)
        mem_file = os.environ.get("MEMORY_FILENAME", globals().get("MEMORY_FILENAME", "kv_memory.jsonl"))
        mem_path = memory_dir / mem_file
        self.memory = _MemClass(mem_path)

        # --- Qwen Omni (audio agent)
        from transformers import Qwen2_5OmniProcessor, Qwen2_5OmniThinkerForConditionalGeneration
        from qwen_omni_utils import process_mm_info

        audio_llm_id = os.environ.get("AUDIO_LLM_ID", "Qwen/Qwen2.5-Omni-7B")
        selector = ModelSelector()
        local_dir = Path(selector.format_model_path(audio_llm_id))
        local_dir.mkdir(parents=True, exist_ok=True)

        self.q_processor = Qwen2_5OmniProcessor.from_pretrained(local_dir, trust_remote_code=True)
        self.q_model = Qwen2_5OmniThinkerForConditionalGeneration.from_pretrained(
            local_dir,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto",
            trust_remote_code=True,
            low_cpu_mem_usage=True,
        ).eval()

        # OPTIONAL: try to offload parts to CPU if you still OOM (requires accelerate)
        try:
            from accelerate import infer_auto_device_map, dispatch_model
            max_memory = {
                0: os.environ.get("QWEN_GPU_MAX_MEM", "18GiB"),   # leave headroom
                "cpu": os.environ.get("QWEN_CPU_MAX_MEM", "48GiB"),
            }
            device_map = infer_auto_device_map(
                self.q_model,
                max_memory=max_memory,
                no_split_module_classes=["Qwen2_5OmniAudioEncoder", "Qwen2_5OmniAudioEncoderLayer"],
            )
            self.q_model = dispatch_model(self.q_model, device_map=device_map)
        except Exception:
            pass

        class _QwenAdapter:
            """
            Minimal adapter with VRAM-friendly audio handling:
            - cap to N clips, crop to <= M seconds, resample to 16 kHz
            - pass None (not []) for empty modalities
            - greedy decode with small max_new_tokens
            """
            MAX_AUDIO_CLIPS = int(os.environ.get("AUDIO_CLIPS_MAX", "2"))
            MAX_AUDIO_SEC   = float(os.environ.get("AUDIO_CLIP_MAX_SEC", "12"))
            TARGET_SR       = int(os.environ.get("AUDIO_TARGET_SR", "16000"))

            def __init__(self, proc, model):
                self.processor = proc
                self.model = model

            @staticmethod
            def _sanitize(txt: str) -> str:
                import re
                keep = []
                for line in txt.splitlines():
                    if re.match(r"^\s*(Human:|User:|Assistant:|System:)\b", line, flags=re.IGNORECASE):
                        continue
                    keep.append(line)
                txt2 = "\n".join(keep).strip()
                return re.sub(r"^(?:\d+\s*)?(?:Human:|User:)\s*", "", txt2, flags=re.IGNORECASE).strip()

            @staticmethod
            def _as_list(x):
                if x is None:
                    return []
                return list(x) if not isinstance(x, (list, tuple)) else list(x)

            @staticmethod
            def _none_if_empty(x):
                return None if (x is None or len(x) == 0) else x

            @staticmethod
            def _resample_to(sr_from: int, wav: np.ndarray, sr_to: int) -> np.ndarray:
                if sr_from == sr_to:
                    return wav.astype(np.float32, copy=False)
                try:
                    import torchaudio
                    t = torch.as_tensor(wav, dtype=torch.float32).unsqueeze(0)
                    t2 = torchaudio.functional.resample(t, sr_from, sr_to)
                    return t2.squeeze(0).cpu().numpy().astype(np.float32)
                except Exception:
                    # linear fallback
                    n_to = int(round(len(wav) * sr_to / sr_from))
                    x = np.linspace(0, 1, num=len(wav), endpoint=False, dtype=np.float64)
                    y = np.interp(np.linspace(0, 1, num=n_to, endpoint=False), x, wav.astype(np.float64))
                    return y.astype(np.float32)

            def answer(self, question: str, audio_hits: list, return_audio: bool = False) -> dict:
                import numpy as np
                import soundfile as sf
                from qwen_omni_utils import process_mm_info

                # Build conversation with at most N cropped/resampled clips
                user_content = [{"type": "text", "text": f"Question: {question}"}]
                clips_used = 0
                for h in (audio_hits or []):
                    if clips_used >= self.MAX_AUDIO_CLIPS:
                        break
                    audio_full, sr = sf.read(h["wav_path"])
                    if audio_full.ndim == 2:
                        audio_full = audio_full.mean(axis=1)
                    i0, i1 = int(h["start_s"] * sr), int(h["end_s"] * sr)
                    i0 = max(0, min(i0, len(audio_full)))
                    i1 = max(i0, min(i1, len(audio_full)))
                    if i1 <= i0:
                        continue
                    seg = audio_full[i0:i1].astype(np.float32, copy=False)

                    # crop to ≤ MAX_AUDIO_SEC and resample to 16k to shrink features
                    max_len = int(self.MAX_AUDIO_SEC * sr)
                    if len(seg) > max_len:
                        seg = seg[:max_len]
                    seg = self._resample_to(sr, seg, self.TARGET_SR)

                    user_content.append({"type": "audio", "audio": seg, "sampling_rate": self.TARGET_SR})
                    clips_used += 1

                conv = [
                    {
                        "role": "system",
                        "content": [
                            {
                                "type": "text",
                                "text": (
                                    "You are a precise analyst. Answer ONLY using the provided audio clips. "
                                    "If the clips do not contain the answer, reply exactly: NOT_FOUND_IN_AUDIO."
                                ),
                            }
                        ],
                    },
                    {"role": "user", "content": user_content},
                ]

                # 1) chat template
                text = self.processor.apply_chat_template(conv, add_generation_prompt=True, tokenize=False)

                # 2) multimodal blobs
                audios, images, videos = process_mm_info(conv, use_audio_in_video=False)
                audios = self._as_list(audios)
                images = self._as_list(images)
                videos = self._as_list(videos)

                # pass None for empty modalities (prevents IndexError)
                audios = self._none_if_empty(audios)
                images = self._none_if_empty(images)
                videos = self._none_if_empty(videos)

                # 3) pack tensors
                inputs = self.processor(
                    text=text,
                    audio=audios,
                    images=images,
                    videos=videos,
                    return_tensors="pt",
                    padding=True,
                    use_audio_in_video=False,
                ).to(self.model.device)

                tok = getattr(self.processor, "tokenizer", None)
                eos_id = getattr(tok, "eos_token_id", None)
                pad_id = getattr(tok, "pad_token_id", eos_id)

                # stop on chat-end if available
                try:
                    im_end_id = tok.convert_tokens_to_ids("<|im_end|>")
                    eos_ids = list({i for i in [eos_id, im_end_id] if i is not None}) or None
                except Exception:
                    eos_ids = eos_id

                # block role tags
                bad_words_ids = None
                try:
                    bad_words = ["Human:", "User:", "Assistant:", "System:"]
                    enc = [tok(bw, add_special_tokens=False).input_ids for bw in bad_words if tok is not None]
                    bad_words_ids = [ids for ids in enc if ids] or None
                except Exception:
                    pass

                torch.cuda.empty_cache()
                with torch.no_grad():
                    gen = self.model.generate(
                        **inputs,
                        use_audio_in_video=False,
                        max_new_tokens=int(os.environ.get("QWEN_MAX_NEW_TOKENS", "96")),
                        do_sample=False,
                        num_beams=1,
                        repetition_penalty=1.05,
                        eos_token_id=eos_ids,
                        pad_token_id=pad_id,
                       # bad_words_ids=bad_words_ids,
                        return_dict_in_generate=True,
                    )

                # robust decode
                seqs = getattr(gen, "sequences", None)
                if seqs is None:
                    answer = ""
                else:
                    prompt_len = inputs["input_ids"].shape[1] if "input_ids" in inputs else 0
                    try:
                        new_tokens = seqs[:, prompt_len:]
                    except Exception:
                        new_tokens = seqs

                    decoded = self.processor.batch_decode(
                        new_tokens,
                        skip_special_tokens=True,
                        clean_up_tokenization_spaces=True,
                    )
                    if decoded:
                        answer = decoded[0].strip()
                    else:
                        decoded_full = self.processor.batch_decode(
                            seqs,
                            skip_special_tokens=True,
                            clean_up_tokenization_spaces=True,
                        )
                        answer = decoded_full[0].strip() if decoded_full else ""

                answer = self._sanitize(answer)
                if answer.strip().upper().startswith("NOT_FOUND_IN_AUDIO"):
                    answer = "Not found in audio."

                evid = [
                    {
                        "file_name": h["file_name"],
                        "file_path": h["file_path"],
                        "start_s": h["start_s"],
                        "end_s": h["end_s"],
                        "score": h.get("score_mmr", h.get("score", 0.0)),
                    }
                    for h in (audio_hits or [])[:self.MAX_AUDIO_CLIPS]
                ]
                return {"answer": answer, "evidence": evid}

        self.audio_llm = _QwenAdapter(self.q_processor, self.q_model)

        # --- Compile graph bound to these artifacts
        self.graph = build_audio_agentic_graph_no_textllm(
            relevance_threshold=self.relevance_threshold,
            fetch_k=self.fetch_k,
            top_k=self.top_k,
            vecs=self.vecs,
            metas=self.metas,
        )

        # Thin wrappers available in predict
        self._mem_get = lambda key: self.memory.get(key)
        self._mem_set = lambda key, val: self.memory.set(key, val)

    def _invoke(self, question: str, file_id: str = "global") -> dict:
        state = self.graph.invoke({
            "question": question,
            "file_id": file_id,
            "memory": self.memory,
            "audio_llm": self.audio_llm,
            "messages": [],
        })
        return state

    def predict(self, context, model_input):
        """
        Accepts:
          - List[dict]: [{"question": "...", "file_id": "..."}, ...]
          - or a pandas.DataFrame with 'question' and optional 'file_id' columns.
        Returns: List[dict] with {answer, evidence, from_memory}.
        """
        if isinstance(model_input, pd.DataFrame):
            records = model_input.to_dict(orient="records")
        elif isinstance(model_input, list):
            records = model_input
        else:
            raise ValueError("Unsupported input. Pass a list of dicts or a pandas DataFrame.")

        out = []
        for r in records:
            q   = (r.get("question") or "").strip()
            fid = r.get("file_id") or "global"
            s = self._invoke(q, fid)
            out.append({
                "question": q,
                "file_id": fid,
                "answer": s.get("answer", ""),
                "evidence": s.get("evidence", []),
                "from_memory": s.get("from_memory", False),
            })
        return out


/opt/conda/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:168: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


In [42]:
import mlflow

# mlflow.set_tracking_uri(MLFLOW_TRACKING)
mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI", "/phoenix/mlflow"))
mlflow.set_experiment(EXPERIMENT_NAME)
print(f"Using MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment: {EXPERIMENT_NAME}")

# Read requirements from your repo (or pin a minimal list inline)
req_path = project_root / "requirements.txt"
if req_path.exists():
    with open(req_path, "r") as f:
        pip_reqs = [ln.strip() for ln in f if ln.strip() and not ln.strip().startswith("#")]
else:
    pip_reqs = [
        "mlflow>=2.10.0",
        "langgraph>=0.2.0",
        "transformers>=4.41.0",
        "torch>=2.1.0",
        "faiss-cpu>=1.7.4",
        "soundfile>=0.12.1",
        "huggingface_hub>=0.23.0",
        "tabulate>=0.9.0",
    ]

with mlflow.start_run(run_name=f"register-{MODEL_NAME}") as run:
    # Artifacts mapping for pyfunc
    artifacts = {
        "index_dir": str(ARTIF_DIR / "index"),
        "config_path": str(ARTIF_DIR / "config" / "config.json"),
        "memory_dir": str(ARTIF_DIR / "memory"),
    }

    # Include local src so the server can import utils, model_selection, etc.
    code_paths = [str(SRC_DIR)]

    model_info = mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=AudioAgenticPyFunc(),
        artifacts=artifacts,
        code_path=code_paths,
        pip_requirements=pip_reqs,
        registered_model_name=MODEL_NAME,
    )

print("Logged model at:", model_info.model_uri)
print("Registered name:", MODEL_NAME)


Using MLflow tracking URI: /phoenix/mlflow
Experiment: AIStudio-Agentic-Audio-RAG-with-LangGraph-Experiment


2025/08/19 21:31:02 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Registered model 'AIStudio-Agentic-Audio-RAG-with-LangGraph-Model' already exists. Creating a new version of this model...


Logged model at: runs:/98642f4bb13a46ec97d02b4625926ca4/model
Registered name: AIStudio-Agentic-Audio-RAG-with-LangGraph-Model


Created version '8' of model 'AIStudio-Agentic-Audio-RAG-with-LangGraph-Model'.


In [43]:
loaded = mlflow.pyfunc.load_model(model_info.model_uri)

TEST_Q = "What is the main idea of the content?"
payload = [{"question": TEST_Q, "file_id": "global"}]

res = loaded.predict(payload)
print(json.dumps(res, indent=2)[:1200], "...")


Unrecognized keys in `rope_scaling` for 'rope_type'='default': {'mrope_section'}


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 3.96 GiB. GPU 0 has a total capacity of 24.00 GiB of which 0 bytes is free. Including non-PyTorch memory, this process has 17179869184.00 GiB memory in use. Of the allocated memory 21.97 GiB is allocated by PyTorch, and 115.41 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# # 1. Set MLflow tracking URI and experiment
# mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI", "/phoenix/mlflow"))
# mlflow.set_experiment(experiment_name=EXPERIMENT_NAME)
# print(f"Using MLflow tracking URI: {mlflow.get_tracking_uri()}")
# print(f"Experiment: {EXPERIMENT_NAME}")

Using MLflow tracking URI: /phoenix/mlflow
Experiment: AIStudio-Agentic-Customer-Feedback-Analyzer-with-LangGraph-Experiment


In [ ]:
# %%time

# # These should point to the actual files you're using for model and memory
# MODEL_ARTIFACTS = {
#     "model_path": str(MODEL_PATH),
#     "memory_path": str(MEMORY_PATH),
# }
 
# # === Start MLflow run, log, and register ===
# with mlflow.start_run(run_name=RUN_NAME) as run:
#     print(f"🚀 Started MLflow run: {run.info.run_id}")

#     # Log and register the model using the classmethod
#     AgenticFeedbackModel.log_model(
#         model_name=MODEL_NAME,
#         model_artifacts=MODEL_ARTIFACTS
#     )

# logger.info(f"✅ Model '{MODEL_NAME}' successfully logged and registered.")

2025/08/02 06:43:49 INFO mlflow.models.signature: Inferring model signature from type hints


🚀 Started MLflow run: e015ccbb3a024c3e9ab35a177ab9d238


Successfully registered model 'AIStudio-Agentic-Customer-Feedback-Analyzer-with-LangGraph-Model'.
Created version '1' of model 'AIStudio-Agentic-Customer-Feedback-Analyzer-with-LangGraph-Model'.


CPU times: user 1 s, sys: 15.7 s, total: 16.7 s
Wall time: 4min 3s


In [ ]:
# # 3. Retrieve the latest version from the Model Registry
# client = MlflowClient()
# versions = client.get_latest_versions(MODEL_NAME, stages=["None"])

# if not versions:
#     raise RuntimeError(f"No registered versions found for model '{MODEL_NAME}'.")
    
# latest_version = versions[0].version
# model_info = mlflow.models.get_model_info(f"models:/{MODEL_NAME}/{latest_version}")

# logger.info(f"Latest registered version of '{MODEL_NAME}': {latest_version}")
# logger.info(f"Signature: {model_info.signature}")

In [ ]:
# %%time

# # 4. Load the model from the Model Registry
# loaded_model = mlflow.pyfunc.load_model(model_uri=f"models:/{MODEL_NAME}/{latest_version}")
# logger.info(f"Successfully loaded model '{MODEL_NAME}' version {latest_version} for inference.")

CPU times: user 1.22 s, sys: 2.34 s, total: 3.56 s
Wall time: 1min 13s


In [ ]:
# # 5. Run a sample inference using the loaded model (Audio RAG)

# from pathlib import Path

# # Collect media files (same extensions you used in the ingestion cell)
# _MEDIA_EXTS = {".mp3", ".wav", ".ogg", ".flac", ".m4a", ".mp4", ".mov", ".avi", ".mkv", ".m4v", ".webm"}
# sample_media_paths = [
#     str(p) for p in sorted(Path(INPUT_PATH).rglob("*"))
#     if p.is_file() and p.suffix.lower() in _MEDIA_EXTS
# ]

# if not sample_media_paths:
#     raise FileNotFoundError(f"No audio/video files found in {INPUT_PATH}. "
#                             f"Please add at least one media file to run a sample inference.")

# # The audio RAG model expects {"paths": [...], "query": "..."}
# input_payload = [{
#     "paths": sample_media_paths,
#     "query": QUESTION  # reuse your QUESTION var, or set a literal test query here
# }]

# print("\n=== Running Sample Inference (Audio RAG) ===")
# results = loaded_model.predict(input_payload)       # preserve list-in / list-out contract
# result = results[0] if isinstance(results, list) else results

# print("Answer:\n", result["answer"])
# print("\nSupporting chunks:")
# for c in result.get("chunks", []):
#     print(f"- [{c['start_s']:.1f}–{c['end_s']:.1f}s] {c['text'][:120]}...")




=== Running Sample Inference ===


🔁 Processing each chunk: 100%|██████████| 2/2 [00:01<00:00,  1.96it/s, group=✅ Chunk 2 response length: 28 chars]


🔁 Processing each grouped chunk answers: 100%|██████████| 1/1 [00:01<00:00,  1.38s/it, group=🧠 Synthesized partial answer (1/1)]



🔚 === Final Answer ===

# 🧠 Synthesized partial answer (1/1)

Since the user's question is about individuals mentioned in the document as providing feedback, and neither Chunk 1 nor Chunk 2 mentions any individuals providing feedback, the final answer is:

**No individuals are mentioned in the document as providing feedback.**




# Generated Answer

In [ ]:
# display(Markdown(result.answer))

# 🧠 Synthesized partial answer (1/1)

Since the user's question is about individuals mentioned in the document as providing feedback, and neither Chunk 1 nor Chunk 2 mentions any individuals providing feedback, the final answer is:

**No individuals are mentioned in the document as providing feedback.**

# Message History

In [ ]:
# print(result.messages)

[
    {
        "role": "developer",
        "content": "User submitted a question."
    },
    {
        "role": "user",
        "content": "Which poeple provided the feedback?"
    },
    {
        "role": "developer",
        "content": "\ud83e\udde0 Relevance check result:"
    },
    {
        "role": "assistant",
        "content": "yes"
    },
    {
        "role": "developer",
        "content": "\ud83e\udded No cached answer found for question: 'Which poeple provided the feedback?'"
    },
    {
        "role": "developer",
        "content": "\u270f\ufe0f Rewritten user question:"
    },
    {
        "role": "assistant",
        "content": "Who are the individuals mentioned in the document as providing feedback?"
    },
    {
        "role": "developer",
        "content": "\ud83e\udde9 Chunked 1 documents into 2 chunks (size=4096, overlap=256)"
    },
    {
        "role": "developer",
        "content": "\ud83e\udde0 Processed 2 chunks for question: 'Who are the individual

In [21]:
end_time: float = time.time()
elapsed_time: float = end_time - start_time
elapsed_minutes: int = int(elapsed_time // 60)
elapsed_seconds: float = elapsed_time % 60

logger.info(f"⏱️ Total execution time: {elapsed_minutes}m {elapsed_seconds:.2f}s")
logger.info("✅ Notebook execution completed successfully.")

Built with ❤️ using [**HP AI Studio**](https://hp.com/ai-studio).